In [ ]:
# ============================================================
# 10A_LSTM_Full_History.ipynb
# Section 1 — Prepare full-history sequences (padding + masking)
# ============================================================
# Ziel / الهدف:
#   Give the LSTM each engine's FULL history (all cycles),
#   not just the last 50 — so it sees the same amount of
#   information as XGBoost(C). Fair comparison.
#
#   Variable-length histories are unified with PRE-padding +
#   a masking layer, so padded zeros are ignored (no fake data).
#   No leakage: test uses only observed cycles.
# ============================================================

import os, sys
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001
N_FEATURES = len(FINAL_FEATURES_FD001)
print(f"Sensors: {N_FEATURES} -> {FINAL_FEATURES_FD001}")

# --- Load frozen processed data (same as every notebook) ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
y_true    = test_last["RUL_true"].values
units_test = test_last["unit"].values

# --- Determine the padding length = longest TRAIN engine ---
train_lengths = train.groupby("unit").size()
test_lengths  = test_full.groupby("unit").size()
MAX_LEN = int(train_lengths.max())
print(f"\nTrain engine lengths: min={train_lengths.min()}, "
      f"max={train_lengths.max()}, mean={train_lengths.mean():.0f}")
print(f"Test  engine lengths: min={test_lengths.min()}, "
      f"max={test_lengths.max()}, mean={test_lengths.mean():.0f}")
print(f"Padding length (MAX_LEN) = {MAX_LEN}")

# Label scaling (same trick used in all sequence models: /125)
Y_MAX = 125.0

# ============================================================
# BUILD FULL-HISTORY TRAINING SAMPLES
# ============================================================
# One sample per (engine, end_cycle): the FULL history from
# cycle 1 up to end_cycle, PRE-padded to MAX_LEN.
# Target = piecewise RUL at end_cycle (scaled by /125).
#
# To keep training size manageable (full history for EVERY
# cycle of every engine ~ 20k samples, each up to 362 long),
# we PRE-pad. Padded rows are zeros and will be masked.
# ============================================================
from tensorflow.keras.preprocessing.sequence import pad_sequences

def build_full_history_train(df, max_len):
    seqs, labels = [], []
    for unit, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        feats = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        pw    = g["RUL_piecewise125"].values.astype(np.float32)
        L = len(g)
        for end in range(L):
            # full history from start to current cycle
            seqs.append(feats[0:end+1])
            labels.append(pw[end])
    X = pad_sequences(seqs, maxlen=max_len, dtype="float32",
                      padding="pre", truncating="pre", value=0.0)
    y = np.array(labels, dtype=np.float32) / Y_MAX
    return X, y

# ---- TEST: one sample per engine = its full observed history ----
def build_full_history_test(df, units_order, max_len):
    seqs = []
    for unit in units_order:
        g = df[df["unit"] == unit].sort_values("cycle")
        feats = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        seqs.append(feats)                       # full observed history
    X = pad_sequences(seqs, maxlen=max_len, dtype="float32",
                      padding="pre", truncating="pre", value=0.0)
    return X

print("\nBuilding full-history sequences...")
X_tr_full, y_tr_full = build_full_history_train(train, MAX_LEN)
X_te_full            = build_full_history_test(test_full, units_test, MAX_LEN)
print(f"  Train: {X_tr_full.shape}  (samples, {MAX_LEN}, {N_FEATURES})")
print(f"  Test:  {X_te_full.shape}")
print(f"  y_train range: [{y_tr_full.min():.3f}, {y_tr_full.max():.3f}] (scaled /125)")

print("\n✓ Section 1 complete: full-history sequences ready.")
print("  Padded PRE (zeros at the start), will be masked in the model.")
print("  No leakage: test uses only observed cycles per engine.")

In [ ]:
# ============================================================
# 10A_LSTM_Full_History.ipynb  (COMPLETE — corrected padding)
# ============================================================
# Give the LSTM each engine's FULL history (all cycles), not
# just the last 50 — so it sees the same information as
# XGBoost(C). Fair comparison.
# Variable lengths unified with POST-padding + masking (cuDNN-safe).
# No leakage: test uses only observed cycles.
# ============================================================

import os, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001
N_FEATURES = len(FINAL_FEATURES_FD001)
print(f"Sensors: {N_FEATURES} -> {FINAL_FEATURES_FD001}")

# --- Load frozen processed data ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
y_true    = test_last["RUL_true"].values
units_test = test_last["unit"].values

train_lengths = train.groupby("unit").size()
MAX_LEN = int(train_lengths.max())
Y_MAX = 125.0
print(f"Padding length (MAX_LEN) = {MAX_LEN}")

# ============================================================
# SECTION 1 — build full-history sequences (POST padding)
# ============================================================
def build_full_history_train(df, max_len):
    seqs, labels = [], []
    for unit, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        feats = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        pw    = g["RUL_piecewise125"].values.astype(np.float32)
        for end in range(len(g)):
            seqs.append(feats[0:end+1])          # full history up to cycle
            labels.append(pw[end])
    X = pad_sequences(seqs, maxlen=max_len, dtype="float32",
                      padding="post", truncating="post", value=0.0)   # POST
    y = np.array(labels, dtype=np.float32) / Y_MAX
    return X, y

def build_full_history_test(df, units_order, max_len):
    seqs = []
    for unit in units_order:
        g = df[df["unit"] == unit].sort_values("cycle")
        seqs.append(g[FINAL_FEATURES_FD001].values.astype(np.float32))
    X = pad_sequences(seqs, maxlen=max_len, dtype="float32",
                      padding="post", truncating="post", value=0.0)   # POST
    return X

print("\nBuilding full-history sequences (POST padding)...")
X_tr_full, y_tr_full = build_full_history_train(train, MAX_LEN)
X_te_full            = build_full_history_test(test_full, units_test, MAX_LEN)
print(f"  Train: {X_tr_full.shape}")
print(f"  Test:  {X_te_full.shape}")
print(f"  y_train range: [{y_tr_full.min():.3f}, {y_tr_full.max():.3f}] (scaled /125)")

# ============================================================
# SECTION 2 — build masked LSTM, train, fair comparison
# ============================================================
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0) - 1.0,
                                         np.exp(d/10.0) - 1.0)))

def build_lstm_full(max_len, n_features):
    model = Sequential([
        Masking(mask_value=0.0, input_shape=(max_len, n_features)),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

model = build_lstm_full(MAX_LEN, N_FEATURES)
model.summary()

es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
print("\nTraining LSTM on FULL history...")
history = model.fit(
    X_tr_full, y_tr_full,
    validation_split=0.2,
    epochs=60, batch_size=64,
    callbacks=[es], verbose=1,
)

y_pred_full = model.predict(X_te_full, verbose=0).flatten() * Y_MAX
y_pred_full = np.clip(y_pred_full, 0, 125)

mae  = mean_absolute_error(y_true, y_pred_full)
rmse = np.sqrt(mean_squared_error(y_true, y_pred_full))
ns   = nasa_score(y_true, y_pred_full)

print("\n" + "=" * 60)
print("LSTM (FULL HISTORY) — RESULT")
print("=" * 60)
print(f"  MAE:        {mae:.2f}")
print(f"  RMSE:       {rmse:.2f}")
print(f"  NASA Score: {ns:.2f}")

# ============================================================
# FAIR COMPARISON TABLE
# ============================================================
print("\n" + "=" * 70)
print("FAIR COMPARISON — every model sees each engine's FULL history")
print("=" * 70)
print(f"{'Model':<20} {'Input form':<26} {'MAE':>7} {'RMSE':>7} {'NASA':>9}")
print("-" * 70)
print(f"{'LR':<20} {'65 features (summary)':<26} {14.99:>7.2f} {17.98:>7.2f} {514.27:>9.2f}")
print(f"{'XGBoost':<20} {'65 features (summary)':<26} {8.57:>7.2f} {12.01:>7.2f} {242.67:>9.2f}")
print(f"{'LSTM (last-50)':<20} {'50 cycles (old, unfair)':<26} {10.69:>7.2f} {14.78:>7.2f} {402.11:>9.2f}")
print(f"{'LSTM (full)':<20} {'all cycles (fair)':<26} {mae:>7.2f} {rmse:>7.2f} {ns:>9.2f}")
print("-" * 70)

pred_df = pd.DataFrame({"unit": units_test, "RUL_true": y_true,
                        "pred_LSTM_full": y_pred_full})
out_path = os.path.join(RESULTS_DIR, "fd001_lstm_full_history.csv")
pred_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
if mae < 8.57:
    print(f"  LSTM(full)={mae:.2f} BEATS XGBoost(features)=8.57")
    print("  -> With equal information, the sequence architecture wins.")
elif mae < 10.69:
    print(f"  LSTM(full)={mae:.2f} improved over LSTM(last-50)=10.69,")
    print(f"     but XGBoost(features)=8.57 still wins.")
    print("  -> Full history helps LSTM, yet simple model + features is best.")
else:
    print(f"  LSTM(full)={mae:.2f} did NOT beat LSTM(last-50)=10.69.")
    print("  -> Full raw history is harder to learn than a good summary.")

In [ ]:
# ============================================================
# 10A — Multi-seed robustness: LSTM (FULL history, fair)
# بنثبت: نتيجة LSTM الكاملة (10.16) ثابتة عبر seeds مختلفة
# العدالة: نفس التاريخ الكامل، نفس المعمارية — الـ seed بس بيتغير
# ============================================================
import os, time
import pandas as pd

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
SEEDS = [42, 43, 44, 45, 46]

lstm_full_runs = []
for seed in SEEDS:
    tf.keras.utils.set_random_seed(seed)
    model = build_lstm_full(MAX_LEN, N_FEATURES)
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                          restore_best_weights=True)
    t0 = time.time()
    model.fit(X_tr_full, y_tr_full, validation_split=0.2,
              epochs=60, batch_size=64, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = np.clip(model.predict(X_te_full, verbose=0).flatten() * Y_MAX, 0, 125)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    lstm_full_runs.append({"seed": seed, "mae": mae, "rmse": rmse,
                           "nasa": ns, "train_s": t_train})
    print(f"  LSTM(full) seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | "
          f"NASA {ns:8.2f} | {t_train:.0f}s")

df_lstm_full = pd.DataFrame(lstm_full_runs)
print(f"\nLSTM (FULL history) across seeds {SEEDS}:")
for m in ["mae", "rmse", "nasa"]:
    print(f"  {m.upper():>5}: {df_lstm_full[m].mean():8.2f} ± {df_lstm_full[m].std():5.2f} "
          f"(min {df_lstm_full[m].min():.2f}, max {df_lstm_full[m].max():.2f})")

df_lstm_full["model"] = "LSTM_full"
df_lstm_full.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_lstm_full.csv"), index=False)
print(f"\nSaved: fd001_multiseed_lstm_full.csv")

In [ ]:
# ============================================================
# NB10A — LSTM on the full observed history, multi-seed
# Change from the earlier version: the prediction of every seed is saved,
# not only that of the first, so the significance analysis can average
# the error across seeds in the same way the reported mean is formed.
# Everything else is unchanged: the engine-level early-stopping split,
# the architecture, and the training settings.
# Self-contained: rebuilds the padded windows from the processed files.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
PRED_DIR      = os.path.join(RESULTS_DIR, "per_seed")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
from preprocessing import FINAL_FEATURES_FD001

os.makedirs(PRED_DIR, exist_ok=True)
SEEDS      = [42, 43, 44, 45, 46]
N_INNER    = 20            # engines held out for early stopping
SPLIT_SEED = 7             # fixed, so every seed sees the same engines
Y_MAX      = 125.0
EPOCHS, BATCH = 60, 128

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

N_FEATURES = len(FINAL_FEATURES_FD001)
MAX_LEN = int(train.groupby("unit").size().max())     # from training only
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values
print(f"Sensors: {N_FEATURES} | MAX_LEN: {MAX_LEN}")

def build_train_windows(df):
    X, y, u = [], [], []
    for unit, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        lab = g["RUL_piecewise125"].values.astype(np.float32)
        for end in range(len(g)):
            seq = f[:end + 1]
            pad = np.zeros((MAX_LEN - len(seq), N_FEATURES), np.float32)
            X.append(np.vstack([seq, pad]))           # post-padding, cuDNN-safe
            y.append(lab[end]); u.append(unit)
    return np.array(X, np.float32), np.array(y, np.float32), np.array(u)

def build_test_windows(df, units_order):
    X = []
    for unit in units_order:
        g = df[df["unit"] == unit].sort_values("cycle")
        seq = g[FINAL_FEATURES_FD001].values.astype(np.float32)[-MAX_LEN:]
        pad = np.zeros((MAX_LEN - len(seq), N_FEATURES), np.float32)
        X.append(np.vstack([seq, pad]))
    return np.array(X, np.float32)

t0 = time.time()
X_all, y_all, u_all = build_train_windows(train)
X_te = build_test_windows(test_full, units_test)
print(f"Built {X_all.shape} train and {X_te.shape} test windows in {time.time()-t0:.0f}s")

# Engine-level inner split for early stopping
rng = np.random.default_rng(SPLIT_SEED)
all_units = np.sort(train["unit"].unique())
inner_val_units = rng.choice(all_units, size=N_INNER, replace=False)
is_val = np.isin(u_all, inner_val_units)
X_fit, y_fit = X_all[~is_val], y_all[~is_val]
X_val, y_val = X_all[is_val],  y_all[is_val]
print(f"Early-stopping split: {len(all_units) - N_INNER} engines "
      f"({len(X_fit)} windows) fit | {N_INNER} engines ({len(X_val)} windows) validate")
print(f"Held-out engines: {sorted(int(u) for u in inner_val_units)}\n")

def build_model():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN, N_FEATURES)),
        layers.Masking(mask_value=0.0),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])

runs, preds = [], {}
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build_model()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(X_fit, y_fit / Y_MAX,
                     validation_data=(X_val, y_val / Y_MAX),
                     epochs=EPOCHS, batch_size=BATCH,
                     callbacks=[es], verbose=0)
    t_train = time.time() - t0

    p = np.clip(model.predict(X_te, verbose=0).ravel() * Y_MAX, 0, Y_MAX)
    preds[seed] = p
    runs.append({"seed": seed,
                 "mae": mean_absolute_error(y_true, p),
                 "rmse": np.sqrt(mean_squared_error(y_true, p)),
                 "nasa": nasa_score(y_true, p),
                 "train_s": t_train,
                 "epochs_run": len(hist.history["loss"])})
    print(f"  seed {seed}: MAE {runs[-1]['mae']:6.2f} | RMSE {runs[-1]['rmse']:6.2f} | "
          f"NASA {runs[-1]['nasa']:8.2f} | {t_train:.0f}s | {runs[-1]['epochs_run']} epochs")

df = pd.DataFrame(runs)
print(f"\nLSTM (full history) across seeds {SEEDS}:")
for m_ in ["mae", "rmse", "nasa"]:
    print(f"  {m_.upper():>5}: {df[m_].mean():8.2f} ± {df[m_].std():5.2f} "
          f"(min {df[m_].min():.2f}, max {df[m_].max():.2f})")
gpus = tf.config.list_physical_devices("GPU")
print(f"  Mean training time: {df['train_s'].mean():.0f}s on "
      f"{gpus[0].name if gpus else 'CPU'}")

df["model"] = "LSTM_full"
df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_lstm_full.csv"), index=False)

wide = pd.DataFrame({"unit": units_test, "RUL_true": y_true})
for seed in SEEDS:
    wide[f"pred_seed{seed}"] = preds[seed]
wide.to_csv(os.path.join(PRED_DIR, "lstm_per_seed.csv"), index=False)

pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_LSTM_full": preds[SEEDS[0]]}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_lstm_full_history.csv"), index=False)

print("\nSaved: fd001_multiseed_lstm_full.csv + per_seed/lstm_per_seed.csv")

# ============================================================
# NB10A — LSTM on the full observed history, multi-seed
# Change from the earlier version: the early-stopping split is now
# defined over engines rather than over rows. Keras' validation_split
# takes the final fraction of the ordered array, and because the windows
# are ordered by engine that cut can fall inside one engine's trajectory,
# so the same engine would appear in both parts. Here twenty engines are
# set aside in advance and never contribute a training window.
# ============================================================
import os, time
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001

SEEDS      = [42, 43, 44, 45, 46]
N_INNER    = 20            # engines held out for early stopping
SPLIT_SEED = 7             # fixed, so every run uses the same inner engines
Y_MAX      = 125.0
EPOCHS, BATCH = 60, 128

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

N_FEATURES = len(FINAL_FEATURES_FD001)
MAX_LEN = int(train.groupby("unit").size().max())     # from training only
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values
print(f"Sensors: {N_FEATURES} | MAX_LEN: {MAX_LEN}")

# --- Build padded windows, keeping the engine id with every window ---
def build_train_windows(df):
    X, y, u = [], [], []
    for unit, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        lab = g["RUL_piecewise125"].values.astype(np.float32)
        for end in range(len(g)):
            seq = f[:end + 1]
            pad = np.zeros((MAX_LEN - len(seq), N_FEATURES), np.float32)
            X.append(np.vstack([seq, pad]))           # post-padding, cuDNN-safe
            y.append(lab[end]); u.append(unit)
    return np.array(X, np.float32), np.array(y, np.float32), np.array(u)

def build_test_windows(df, units_order):
    X = []
    for unit in units_order:
        g = df[df["unit"] == unit].sort_values("cycle")
        seq = g[FINAL_FEATURES_FD001].values.astype(np.float32)[-MAX_LEN:]
        pad = np.zeros((MAX_LEN - len(seq), N_FEATURES), np.float32)
        X.append(np.vstack([seq, pad]))
    return np.array(X, np.float32)

t0 = time.time()
X_all, y_all, u_all = build_train_windows(train)
X_te = build_test_windows(test_full, units_test)
print(f"Built {X_all.shape} train and {X_te.shape} test windows in {time.time()-t0:.0f}s")

# --- Engine-level inner split for early stopping ---
# The split is over engines, so no engine contributes windows to both
# parts. The same engines are used for every seed, so the seeds differ
# only in weight initialisation and not in the data they see.
rng = np.random.default_rng(SPLIT_SEED)
all_units = np.sort(train["unit"].unique())
inner_val_units = rng.choice(all_units, size=N_INNER, replace=False)
is_val = np.isin(u_all, inner_val_units)

X_fit,  y_fit  = X_all[~is_val], y_all[~is_val]
X_val,  y_val  = X_all[is_val],  y_all[is_val]
print(f"Early-stopping split: {len(all_units) - N_INNER} engines "
      f"({len(X_fit)} windows) fit | {N_INNER} engines ({len(X_val)} windows) validate")
print(f"Held-out engines: {sorted(inner_val_units)}")

def build_model():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN, N_FEATURES)),
        layers.Masking(mask_value=0.0),               # ignore the padded tail
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])

runs = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build_model()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(X_fit, y_fit / Y_MAX,
                     validation_data=(X_val, y_val / Y_MAX),
                     epochs=EPOCHS, batch_size=BATCH,
                     callbacks=[es], verbose=0)
    t_train = time.time() - t0

    y_pred = np.clip(model.predict(X_te, verbose=0).ravel() * Y_MAX, 0, Y_MAX)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    runs.append({"seed": seed, "mae": mae, "rmse": rmse, "nasa": ns,
                 "train_s": t_train, "epochs_run": len(hist.history["loss"])})
    print(f"  seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | NASA {ns:8.2f} | "
          f"{t_train:.0f}s | {len(hist.history['loss'])} epochs")
    if seed == SEEDS[0]:
        pd.DataFrame({"unit": units_test, "RUL_true": y_true,
                      "pred_LSTM_full": y_pred}).to_csv(
            os.path.join(RESULTS_DIR, "fd001_lstm_full_history.csv"), index=False)

df = pd.DataFrame(runs)
print(f"\nLSTM (full history) across seeds {SEEDS}:")
for m in ["mae", "rmse", "nasa"]:
    print(f"  {m.upper():>5}: {df[m].mean():8.2f} ± {df[m].std():5.2f} "
          f"(min {df[m].min():.2f}, max {df[m].max():.2f})")
print(f"  Mean training time: {df['train_s'].mean():.0f}s on "
      f"{tf.config.list_physical_devices('GPU')[0].name if tf.config.list_physical_devices('GPU') else 'CPU'}")

df["model"] = "LSTM_full"
df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_lstm_full.csv"), index=False)
print("\nSaved: fd001_multiseed_lstm_full.csv + fd001_lstm_full_history.csv")

In [ ]:
build_model().summary()